# Tara N1 Pretraining

- **Data set:** 500M token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`
- **tara_n1_pretrain.pth:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth`
## Completed training phases:
- ✅Completed first 100M - resulting 100M token corpus
- ✅Completed next 100M - resulting 200M token corpus
- ✅Completed next 100M - resulting 300M token corpus
- ✅Completed next 100M - resulting 400M token corpus
- Completed next 100M - resulting 500M token corpus

In [1]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth")
with open("tara_n1_pretrain.pth", "wb") as f:
    f.write(model_res.content)
print("Downloaded wieghts")

Downloaded model.py successfully.
Downloaded train_utils.py successfully.
Downloaded wieghts


In [3]:
from model import *
from train_utils import *

In [4]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=256,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

In [5]:
# modelV1 = CustomGPT(config)

# if torch.cuda.device_count() > 1:
#     modelV1 = nn.DataParallel(modelV1)
#     print(f"Using {torch.cuda.device_count()} GPUs")

# modelV1.to(device)

# calc_params(modelV1)

# loss_fn = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(modelV1.parameters(), lr=1e-4)
# scaler = torch.amp.GradScaler()


modelV2 = CustomGPT(config)
# state_dict = torch.load("tara_n1_pretrain.pth", map_location=device)
# state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
modelV2.load_weights("tara_n1_pretrain.pth")

if torch.cuda.device_count() > 1:
    modelV2 = nn.DataParallel(modelV2)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV2.to(device)

calc_params(modelV2)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Loaded weights from tara_n1_pretrain.pth
Using 2 GPUs
Total Parameters: 30,586,449
Trainable Parameters: 30,586,449


# The Dataset

In [6]:
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
# target_tokens = 100_000_000
# skip_tokens = 0

# skip_tokens = 100_000_000
# target_tokens = 200_000_000
# fname = "tokens_100m_to_200m.bin" 

# skip_tokens = 200_000_000
# target_tokens = 300_000_000
# fname = "tokens_200m_to_300m.bin"

skip_tokens = 500_000_000
target_tokens = 1_000_000_000
fname = "tokens_300m_to_400m.bin" 

total_tokens = target_tokens - skip_tokens
ds = load_dataset("HuggingFaceFW/fineweb", split="train", name="sample-10BT", streaming=True)

tokens_mmap = np.memmap(
    fname, 
    dtype=np.uint16,
    mode="w+",
    shape=(total_tokens, )
)
seen_tokens = 0
written = 0
pbar = tqdm(total=total_tokens)
for sample in ds:
    text = sample["text"]
    tokenized = tokenizer.encode(text)
    
    if seen_tokens+len(tokenized) <= skip_tokens:
        seen_tokens += len(tokenized)
        continue
        
    # tokens.extend(tokenized)
    n = min(len(tokenized), total_tokens-written)
    tokens_mmap[written : written+n] = np.array(tokenized[:n], dtype=np.uint16)
    written += n
    # curr = min(len(tokens), target_tokens-skip_tokens)
    pbar.n = written
    pbar.update(0)
    # seen_token += len(tokenized)
    if written >= total_tokens:
        break

tokens_mmap.flush()
pbar.close()

tokens = np.memmap(fname, dtype=np.uint16, mode="r", shape=(total_tokens,))
# tokens = tokens_mmap()

print(f"Collected {len(tokens)} tokens.")


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

  0%|          | 0/100000000 [00:00<?, ?it/s]

Collected 100000000 tokens.


In [7]:
train_split = 0.9
train_dataloader, test_dataloader = create_dataloaders(tokens, train_split, device, block_size = config.block_size, batch_size=32)

Data length: 100000000
Train data length: 90000000
Test data length: 10000000
Train batches: 32.00009102248113
Test batches: 32.000819220972055


# Pretraining the model

In [8]:
from tqdm.auto import tqdm
steps = 55000
train_iter = iter(train_dataloader)
for step in tqdm(range(1, steps+1)):
    # modelV1.train()
    modelV2.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    # train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    train_loss, train_iter = train_step(modelV2, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    
    if step % 5000 == 0:
        # modelV1.eval()
        modelV2.eval()
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        # test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        test_loss = test_step(modelV2, test_dataloader, 10, loss_fn, device)
        print(f"Step {step} | Train Loss: {train_loss} | Test Loss: {test_loss:}")

  0%|          | 0/20000 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Step 5000 | Train Loss: 4.745969295501709 | Test Loss: 4.400051307678223


0it [00:00, ?it/s]

Step 10000 | Train Loss: 4.604597091674805 | Test Loss: 4.330773138999939


0it [00:00, ?it/s]

Step 15000 | Train Loss: 4.706794738769531 | Test Loss: 4.379744005203247


0it [00:00, ?it/s]

Step 20000 | Train Loss: 4.411601543426514 | Test Loss: 4.333974385261536


In [9]:
# torch.save(modelV1.state_dict(), "tara_n1_pretrain_v1.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v2.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v3.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v4.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v5.pth")
torch.save(modelV2.state_dict(), "tara_n1_pretrain_v6.pth")

# Testing



In [10]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

# modelV1.eval()
test_model = CustomGPT(config)
test_model.load_weights("tara_n1_pretrain_v6.pth")
# test_model.load_weights("tara_n1_pretrain.pth")
test_model.to(device)
test_model.eval()
with torch.inference_mode():
    # output = modelV1.generate(context, max_new_tokens=100)
    output = test_model.generate(context)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")


Loaded weights from tara_n1_pretrain_v4.pth
Input:
Once upon a time, 

Output:
Once upon a time, urn, theft, etc
As for the DEA Army I was making the Siefers there, and every meteorologist in the present day three years ago, ‘A witness howability, litigation, investigations and investigative planning help us determine how the developments and practices of a multi-year expert detectives’ experience are increasingly sparsely restricted. Most recently, the applicants in the U. Army, are taking into the corporate headquarters in Rhode Island, where they pursue free administration service.
The increased economy (RPS) totaled up to $74,000 for Sales. Funds were $4,000 and up for an employee named Ramella Jillian Housing Inc., California.
On Monday, 58 Holley has worked with NewOrange County Ohio City Council.
Share this article by Elizabeth Lori Hubbards GymCCVP7.HERE
No sooner you Apora’s Fulton Home
Chat about your two days design competition will guide you to the organizers – MIDI songs a